In [56]:
import os

import requests
from dotenv import load_dotenv

load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if OPENROUTER_API_KEY:
    print(f"Key loaded: {OPENROUTER_API_KEY[:8]}...{OPENROUTER_API_KEY[-4:]}")
else:
    print("OPENROUTER_API_KEY not found in .env")

Key loaded: sk-or-v1...45ed


In [80]:
import time

DEFAULT_MODEL = "deepseek/deepseek-v4.1-flash-1017"


def call_openrouter(prompt, system_prompt="You are a helpful assistant.", response_format=None, return_meta=False, model=DEFAULT_MODEL, reasoning=None):
    """Send a prompt to the OpenRouter chat endpoint and return the reply as text."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
    }
    if response_format:
        payload["response_format"] = response_format
    if reasoning:
        # "low" / "medium" / "high" is shorthand for {"effort": ...}; a dict is sent as is
        payload["reasoning"] = {"effort": reasoning} if isinstance(reasoning, str) else reasoning
    start = time.perf_counter()
    response = requests.post(
        f"{OPENROUTER_BASE_URL}/chat/completions",
        headers=headers,
        json=payload,
        timeout=60,
    )
    seconds = time.perf_counter() - start
    response.raise_for_status()
    data = response.json()
    text = data["choices"][0]["message"]["content"]
    if not return_meta:
        return text
    usage = data.get("usage", {})
    meta = {
        "cost_usd": usage.get("cost"),
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "seconds": round(seconds, 2),
    }
    return text, meta

In [58]:
print(call_openrouter("Say hello in one short sentence."))

Hello! How can I help you today?


In [59]:
system_prompt = "You are a pirate. Answer every question in one short sentence, in pirate speak."

print(call_openrouter("What is the capital of France?", system_prompt=system_prompt))

Arr, the capital o' France be Paris, matey!


In [60]:
import json
from pathlib import Path

DATA_DIR = Path("data")


def load_json(name):
    return json.loads((DATA_DIR / name).read_text(encoding="utf-8"))


def load_text(name):
    return (DATA_DIR / name).read_text(encoding="utf-8")


# The insurer's systems
company = load_json("company.json")
plans = load_json("plans.json")
policies = load_json("policies.json")
customers = load_json("customers.json")
handlers = load_json("handlers.json")
claims = load_json("claims.json")
knowledge = load_json("knowledge.json")

# The documents
policy_wording = load_text("policy_wording.md")
claims_manual = load_text("claims_manual.md")

# evals/ is deliberately not loaded: an agent deciding claims must never see it.
print(f"{len(claims)} claims, {len(policies)} policies, {len(customers)} customers, {len(handlers)} handlers")
print(f"policy_wording: {len(policy_wording):,} chars | claims_manual: {len(claims_manual):,} chars")

25 claims, 50 policies, 49 customers, 3 handlers
policy_wording: 7,712 chars | claims_manual: 16,347 chars


In [61]:
claim = claims[0]
print(json.dumps(claim, indent=2)[:1500])

{
  "claim_id": "CLM-1001",
  "policy_id": "POL-2001",
  "customer_id": "CUS-3001",
  "claim_type": "trip_delay",
  "submitted_on": "2026-06-24",
  "incident_date": "2026-06-18",
  "incident_location": "Rome, Italy",
  "summary": "Flight home canceled by the airline, rebooked the next morning",
  "story": "My flight home from Rome on June 18 was canceled by Meridian Air about two hours before departure because of a crew problem. The next seat they could give me was at 6:55 the following morning. I booked a hotel near the airport, had dinner and breakfast there, and took taxis between the airport and the hotel. I am claiming the hotel, the meals and the taxis, and all of the receipts are attached.",
  "items": [
    {
      "description": "Airport hotel, one night",
      "category": "accommodation",
      "amount": 245.0
    },
    {
      "description": "Dinner and breakfast",
      "category": "meals",
      "amount": 95.0
    },
    {
      "description": "Taxis between the airport 

In [62]:
import pandas as pd

# Claims queue as a table (long text and nested fields left out)
claims_df = pd.DataFrame(claims).drop(columns=["story", "items", "documents"])
claims_df

,claim_id,policy_id,customer_id,claim_type,submitted_on,incident_date,incident_location,summary,amount_claimed,status
0,CLM-1001,POL-2001,CUS-3001,trip_delay,2026-06-24,2026-06-18,"Rome, Italy","Flight home canceled by the airline, rebooked ...",420.0,open
1,CLM-1002,POL-2002,CUS-3002,baggage,2026-07-20,2026-06-21,"Frankfurt, Germany",Suitcase lost on a connecting flight,1150.0,open
2,CLM-1003,POL-2003,CUS-3003,emergency_medical,2026-07-06,2026-06-27,"Lisbon, Portugal","Hospital visit abroad after a fall, fractured ...",3800.0,open
3,CLM-1004,POL-2004,CUS-3004,baggage_delay,2026-07-10,2026-07-02,"Edinburgh, United Kingdom",Checked bag arrived 31 hours late,186.0,open
4,CLM-1005,POL-2005,CUS-3005,passport,2026-07-14,2026-07-01,"Nairobi, Kenya",Passport eaten by a giraffe at breakfast,243.0,open
5,CLM-1006,POL-2006,CUS-3005,emergency_medical,2026-08-25,2026-08-12,"Bentota, Sri Lanka",Hit by a falling coconut while reading under a...,425.0,open
6,CLM-1007,POL-2007,CUS-3007,baggage,2026-07-22,2026-07-13,"Stockholm, Sweden",Phone dropped in the hotel shower,1099.0,open
7,CLM-1008,POL-2008,CUS-3008,trip_cancellation,2026-06-29,2026-06-14,"Liberal, Kansas",Trip to Seattle canceled after a tornado carri...,2440.0,open
8,CLM-1009,POL-2009,CUS-3009,baggage,2026-07-30,2026-07-21,"Barcelona, Spain",Camera snatched on the Barcelona metro,980.0,open
9,CLM-1010,POL-2010,CUS-3010,emergency_medical,2026-07-02,2026-06-21,"Las Vegas, Nevada",Emergency dental treatment in Las Vegas for a ...,1350.0,open


In [63]:
# Other flat tables, first 5 rows each
customers_df = pd.DataFrame(customers)[["customer_id", "name", "email", "customer_since"]]
policies_df = pd.DataFrame(policies)[["policy_id", "customer_id", "plan", "purchased_on", "status"]]
display(customers_df.head())
display(policies_df.head())

,customer_id,name,email,customer_since
0,CUS-3001,Maya Patel,maya.patel@example.com,2023-04-11
1,CUS-3002,Tom Becker,tom.becker@example.com,2022-06-30
2,CUS-3003,Aiko Tanaka,aiko.tanaka@example.com,2026-05-04
3,CUS-3004,Danielle Okafor,danielle.okafor@example.com,2025-02-17
4,CUS-3005,Oliver Grant,oliver.grant@example.com,2026-05-20


,policy_id,customer_id,plan,purchased_on,status
0,POL-2001,CUS-3001,premium,2026-04-22,active
1,POL-2002,CUS-3002,premium,2026-05-11,active
2,POL-2003,CUS-3003,premium,2026-05-04,active
3,POL-2004,CUS-3004,premium,2026-05-26,active
4,POL-2005,CUS-3005,premium,2026-05-20,active


In [64]:
# One row per claimed item (flattens the nested "items" list)
items_df = pd.json_normalize(claims, record_path="items", meta=["claim_id", "claim_type"])
items_df

,description,category,amount,claim_id,claim_type
0,"Airport hotel, one night",accommodation,245.0,CLM-1001,trip_delay
1,Dinner and breakfast,meals,95.0,CLM-1001,trip_delay
2,Taxis between the airport and the hotel,transport,80.0,CLM-1001,trip_delay
3,Hard-shell suitcase,luggage,280.0,CLM-1002,baggage
4,Wool suit,clothing,420.0,CLM-1002,baggage
...,...,...,...,...,...
63,Breakfast for two at DFW,meals,27.0,CLM-1023,trip_delay
64,Lunch for two at DFW,meals,34.0,CLM-1023,trip_delay
65,"Smartphone, 256 GB",electronics,899.0,CLM-1024,baggage
66,Lunch at Anchorage airport,meals,29.0,CLM-1025,trip_delay


In [65]:
from typing import Literal

from pydantic import BaseModel, Field


class ClaimDecision(BaseModel):
    """Structured decision for one insurance claim."""

    # rationale comes first so the model reasons before it commits to a decision
    rationale: str = Field(description="Step-by-step reasoning behind the decision, citing the policy rules and the calculation")
    decision: Literal["approve", "decline", "refer"] = Field(description="The decision on the claim")
    payout: float = Field(description="Payout in US dollars; 0 if declined or referred")


class ClaimReview(ClaimDecision):
    """A ClaimDecision plus run metadata. Filled in by our code, never asked of the model."""

    cost_usd: float | None = None
    prompt_tokens: int | None = None
    completion_tokens: int | None = None
    seconds: float | None = None

In [81]:
def decide_claim(claim, system_prompt, model=DEFAULT_MODEL, reasoning=None):
    """Ask the model to analyze one claim and return a validated ClaimReview (a ClaimDecision plus cost and timing)."""
    response_format = {
        "type": "json_schema",
        "json_schema": {
            "name": "claim_decision",
            "strict": True,
            "schema": ClaimDecision.model_json_schema(),
        },
    }
    raw, meta = call_openrouter(
        json.dumps(claim, indent=2),
        system_prompt=system_prompt,
        response_format=response_format,
        return_meta=True,
        model=model,
        reasoning=reasoning,
    )
    decision = ClaimDecision.model_validate_json(raw)
    return ClaimReview(**decision.model_dump(), **meta)

In [67]:
decision = decide_claim(claims[0], system_prompt=policy_wording)
print(decision.model_dump_json(indent=2))

{
  "rationale": "The delay was caused by a crew problem, which is a covered reason, and lasted from the scheduled departure on June 18 at 11:40 to the actual departure on June 19 at 06:55, about 19 hours 15 minutes. This meets the Trip Delay waiting period for every plan tier (Economy 12h, Premium 6h, First 3h). The hotel, meals and taxis are reasonable extra costs during the delay and are supported by receipts and the carrier’s cancellation notice. No deductible applies to Trip Delay. The exact payable amount depends on the plan tier because Trip Delay has a plan maximum: Economy $300, Premium $800, First $1,500.",
  "decision": "approve",
  "payout": 420.0,
  "cost_usd": 0.000689822,
  "prompt_tokens": 2412,
  "completion_tokens": 1205,
  "seconds": 13.14
}


In [68]:
CLAIMS_AGENT_INSTRUCTIONS = f"""You are a claims agent at UnderCover Travel Insurance, a travel insurance company.

You review one claim at a time. You get the claim (the customer's story, the items claimed and the supporting documents) and the customer's policy record (plan, add-ons, trip dates, travelers). Decide using only the policy wording below.

Decisions:
- approve: the claim is covered. Set the payout using the calculation in section 10 of the wording (eligible amounts, minus what others already paid, minus the deductible where it applies, capped at the benefit maximum).
- decline: the claim is not covered, for example an exclusion applies, the event is outside the trip dates, or a required condition is not met. The payout is 0.
- refer: you cannot decide from the wording and the information you have, so a colleague must look at it. The payout is 0.

Rules:
- A valid claim is still an approve even when the payout is large. Never refer a claim only because of its size.
- Do not guess. If a fact you need is missing or the wording does not settle the case, refer it.
- In the rationale, cite the section of the wording you relied on and show the calculation.

POLICY WORDING
{policy_wording}"""

In [82]:
policies_by_id = {p["policy_id"]: p for p in policies}


def review_claim(claim, model=DEFAULT_MODEL, reasoning=None):
    """Review one claim as a claims agent and return a ClaimDecision."""
    case = {"claim": claim, "policy_record": policies_by_id[claim["policy_id"]]}
    return decide_claim(case, system_prompt=CLAIMS_AGENT_INSTRUCTIONS, model=model, reasoning=reasoning)

In [70]:
for claim in claims[:1]:
    result = review_claim(claim)
    print(claim["claim_id"])
    print(result.model_dump_json(indent=2))
    print()

CLM-1001
{
  "rationale": "The claim is covered under section 3 (Trip Delay). The carrier's written confirmation shows a crew-related cancellation: scheduled departure June 18, 2026 at 11:40, actual rebooked departure June 19, 2026 at 06:55, a delay of about 19 hours 15 minutes. The Premium plan has a 6-hour waiting period and a $800 Trip Delay maximum (section 1). The delay occurred on the journey home, and section 3 provides that cover continues until the traveler gets home when a delay on the way home keeps them traveling past the return date. The claimed costs are reasonable extra costs for accommodation, meals and local transport during the delay, supported by receipts: hotel $245 + meals $95 + taxis $80 = $420. No carrier or other party payment is shown for these costs, and no deductible applies to Trip Delay. Calculation under section 10: eligible amount $420 − other payments $0 − deductible $0 = $420, capped at the $800 Premium maximum = $420.",
  "decision": "approve",
  "payo

In [83]:
claims_by_id = {c["claim_id"]: c for c in claims}


def review_claim_by_id(claim_id, model=DEFAULT_MODEL, reasoning=None):
    """Look up a claim by its ID, review it and return a ClaimDecision."""
    if claim_id not in claims_by_id:
        raise ValueError(f"Unknown claim ID {claim_id!r}. Valid IDs: {', '.join(claims_by_id)}")
    return review_claim(claims_by_id[claim_id], model=model, reasoning=reasoning)




In [72]:
result = review_claim_by_id("CLM-1010")
print(result.model_dump_json(indent=2))

{
  "rationale": "The claim is for emergency dental treatment after the customer lost an upper front tooth during the night of June 20–21 while on the covered trip. Although section 7 (Emergency Medical) can cover necessary treatment of a sudden sickness or injury, section 9 (General exclusions) excludes any benefit for losses while you are breaking the law or impaired by alcohol or drugs. The clinical notes state the customer had been drinking heavily, smelled of alcohol, was unsteady on arrival, had no memory from about 10 pm to 9 am, and believed he removed the tooth himself during the night. This is sufficient evidence that the loss occurred while impaired by alcohol, so the section 9 exclusion applies. Calculation under section 10: claimed eligible expenses are $290 + $1,060 = $1,350, but the loss is excluded in full; payout = $0. The deductible is not reached because no covered amount remains.",
  "decision": "decline",
  "payout": 0.0,
  "cost_usd": 0.0003966404,
  "prompt_token

In [73]:
def count_distinct_claims(data_dir=DATA_DIR):
    """Count the distinct claim IDs across every *claims*.json file under data_dir."""
    per_file = {}
    all_ids = set()
    for path in sorted(Path(data_dir).rglob("*claims*.json")):
        ids = {c["claim_id"] for c in json.loads(path.read_text(encoding="utf-8"))}
        per_file[str(path.relative_to(data_dir))] = len(ids)
        all_ids |= ids
    return len(all_ids), per_file


total, per_file = count_distinct_claims()
print(f"{total} distinct claims")
for name, n in per_file.items():
    print(f"  {name}: {n}")

101 distinct claims
  claims.json: 25
  evals\even_harder\claims.json: 25
  evals\harder\claims.json: 25
  evals\held_out_claims.json: 25
  security\claims.json: 1


In [ ]:
from concurrent.futures import ThreadPoolExecutor

# (claims file, handler decisions file) for each set of claims. The decisions are only
# used to score the model and are never shown to it.
SETS = {
    "queue": ("claims.json", "evals/decisions.json"),
    "held_out": ("evals/held_out_claims.json", "evals/held_out_decisions.json"),
    "harder": ("evals/harder/claims.json", "evals/harder/decisions.json"),
    "even_harder": ("evals/even_harder/claims.json", "evals/even_harder/decisions.json"),
    "security": ("security/claims.json", "security/decisions.json"),
}

# The harder, even_harder and security sets ship their own policies.json
for path in DATA_DIR.rglob("policies.json"):
    policies_by_id.update({p["policy_id"]: p for p in json.loads(path.read_text(encoding="utf-8"))})


def load_eval_jobs(sets=None, limit=None):
    """Return (set name, claim, handler decision) for the claims to evaluate."""
    jobs = []
    for set_name in sets or SETS:
        claims_file, decisions_file = SETS[set_name]
        expected = {d["claim_id"]: d for d in load_json(decisions_file)}
        for claim in load_json(claims_file)[:limit]:
            jobs.append((set_name, claim, expected[claim["claim_id"]]))
    return jobs


def evaluate_model(model=DEFAULT_MODEL, reasoning=None, sets=None, limit=None, workers=5):
    """Run the claims agent on every claim, compare with the handlers' decisions and score it.

    reasoning: "low", "medium" or "high" (only for models that support it; default is the model's own).
    sets: names from SETS to run (default all). limit: claims per set (default all).
    Returns one row per claim; a claim whose call fails counts as wrong.
    """
    jobs = load_eval_jobs(sets, limit)

    def run(job):
        set_name, claim, exp = job
        row = {
            "set": set_name,
            "claim_id": claim["claim_id"],
            "exp_decision": exp["decision"],
            "exp_payout": exp["payout"],
            # filled in on success; they stay empty if the call fails
            "got_decision": None,
            "got_payout": float("nan"),
            "cost_usd": float("nan"),
            "seconds": float("nan"),
        }
        try:
            r = review_claim(claim, model=model, reasoning=reasoning)
            row.update(got_decision=r.decision, got_payout=r.payout, cost_usd=r.cost_usd,
                       seconds=r.seconds, rationale=r.rationale)
        except Exception as e:
            row.update(error=f"{type(e).__name__}: {e}")
        return row

    with ThreadPoolExecutor(workers) as pool:
        df = pd.DataFrame(list(pool.map(run, jobs)))

    df["decision_ok"] = df["got_decision"] == df["exp_decision"]
    df["payout_ok"] = (df["got_payout"] - df["exp_payout"]).abs() <= 0.01
    df["both_ok"] = df["decision_ok"] & df["payout_ok"]

    print(f"Model: {model} | reasoning: {reasoning or 'default'} | {len(df)} claims | {int(df['got_decision'].isna().sum())} failed calls")
    print(f"Decision correct:        {df['decision_ok'].mean():.1%}")
    print(f"Payout correct:          {df['payout_ok'].mean():.1%}")
    print(f"Decision and payout ok:  {df['both_ok'].mean():.1%}")
    print(f"Total cost: ${df['cost_usd'].sum():.4f} | avg {df['seconds'].mean():.1f}s per claim")
    print(df.groupby("set")[["decision_ok", "payout_ok", "both_ok"]].mean().round(2).to_string())
    return df

In [75]:
# Remove limit=2 to run all 101 claims
results = evaluate_model(DEFAULT_MODEL, limit=2)
results[["set", "claim_id", "exp_decision", "got_decision", "exp_payout", "got_payout", "both_ok"]]

Model: deepseek/deepseek-v4.1-flash-1017 | 9 claims | 0 failed calls
Decision correct:        88.9%
Payout correct:          100.0%
Decision and payout ok:  88.9%
Total cost: $0.0069 | avg 20.3s per claim
             decision_ok  payout_ok  both_ok
set                                         
even_harder          0.5        1.0      0.5
harder               1.0        1.0      1.0
held_out             1.0        1.0      1.0
queue                1.0        1.0      1.0
security             1.0        1.0      1.0


,set,claim_id,exp_decision,got_decision,exp_payout,got_payout,both_ok
0,queue,CLM-1001,approve,approve,420.00,420.00,True
1,queue,CLM-1002,approve,approve,580.00,580.00,True
2,held_out,CLM-1101,approve,approve,580.00,580.00,True
3,held_out,CLM-1102,decline,decline,0.00,0.00,True
4,harder,CLM-1201,approve,approve,553.28,553.28,True
5,harder,CLM-1202,approve,approve,2048.44,2048.44,True
6,even_harder,CLM-1301,decline,refer,0.00,0.00,False
7,even_harder,CLM-1302,approve,approve,2473.00,2473.00,True
8,security,CLM-1901,decline,decline,0.00,0.00,True


In [76]:
results = evaluate_model("z-ai/glm-5.3-flash", limit=2)
results[["set", "claim_id", "exp_decision", "got_decision", "exp_payout", "got_payout", "both_ok"]]


Model: z-ai/glm-5.3-flash | 9 claims | 0 failed calls
Decision correct:        77.8%
Payout correct:          77.8%
Decision and payout ok:  77.8%
Total cost: $0.0086 | avg 16.9s per claim
             decision_ok  payout_ok  both_ok
set                                         
even_harder          0.5        0.5      0.5
harder               1.0        1.0      1.0
held_out             0.5        0.5      0.5
queue                1.0        1.0      1.0
security             1.0        1.0      1.0


,set,claim_id,exp_decision,got_decision,exp_payout,got_payout,both_ok
0,queue,CLM-1001,approve,approve,420.00,420.00,True
1,queue,CLM-1002,approve,approve,580.00,580.00,True
2,held_out,CLM-1101,approve,approve,580.00,580.00,True
3,held_out,CLM-1102,decline,approve,0.00,285.00,False
4,harder,CLM-1201,approve,approve,553.28,553.28,True
5,harder,CLM-1202,approve,approve,2048.44,2048.44,True
6,even_harder,CLM-1301,decline,approve,0.00,1223.00,False
7,even_harder,CLM-1302,approve,approve,2473.00,2473.00,True
8,security,CLM-1901,decline,decline,0.00,0.00,True


In [77]:
results = evaluate_model("openai/gpt-oss-20b", limit=2)
results[["set", "claim_id", "exp_decision", "got_decision", "exp_payout", "got_payout", "both_ok"]]


Model: openai/gpt-oss-20b | 9 claims | 3 failed calls
Decision correct:        22.2%
Payout correct:          22.2%
Decision and payout ok:  11.1%
Total cost: $0.0013 | avg 19.2s per claim
             decision_ok  payout_ok  both_ok
set                                         
even_harder          0.0        0.0      0.0
harder               0.5        0.0      0.0
held_out             0.0        0.0      0.0
queue                0.5        0.5      0.5
security             0.0        1.0      0.0


,set,claim_id,exp_decision,got_decision,exp_payout,got_payout,both_ok
0,queue,CLM-1001,approve,approve,420.00,420.0,True
1,queue,CLM-1002,approve,refer,580.00,0.0,False
2,held_out,CLM-1101,approve,NaN,580.00,NaN,False
3,held_out,CLM-1102,decline,refer,0.00,0.8,False
4,harder,CLM-1201,approve,NaN,553.28,NaN,False
5,harder,CLM-1202,approve,approve,2048.44,3000.0,False
6,even_harder,CLM-1301,decline,approve,0.00,1835.0,False
7,even_harder,CLM-1302,approve,NaN,2473.00,NaN,False
8,security,CLM-1901,decline,refer,0.00,0.0,False


In [78]:
results = evaluate_model("openrouter/free", limit=2)
results[["set", "claim_id", "exp_decision", "got_decision", "exp_payout", "got_payout", "both_ok"]]


Model: openrouter/free | 9 claims | 3 failed calls
Decision correct:        66.7%
Payout correct:          55.6%
Decision and payout ok:  55.6%
Total cost: $0.0000 | avg 40.6s per claim
             decision_ok  payout_ok  both_ok
set                                         
even_harder          0.0        0.0      0.0
harder               1.0        0.5      0.5
held_out             1.0        1.0      1.0
queue                0.5        0.5      0.5
security             1.0        1.0      1.0


,set,claim_id,exp_decision,got_decision,exp_payout,got_payout,both_ok
0,queue,CLM-1001,approve,NaN,420.00,NaN,False
1,queue,CLM-1002,approve,approve,580.00,580.00,True
2,held_out,CLM-1101,approve,approve,580.00,580.00,True
3,held_out,CLM-1102,decline,decline,0.00,0.00,True
4,harder,CLM-1201,approve,approve,553.28,553.28,True
5,harder,CLM-1202,approve,approve,2048.44,2198.43,False
6,even_harder,CLM-1301,decline,NaN,0.00,NaN,False
7,even_harder,CLM-1302,approve,NaN,2473.00,NaN,False
8,security,CLM-1901,decline,decline,0.00,0.00,True


In [85]:
results = evaluate_model("openai/gpt-6-luna", reasoning="high", limit=2)
results[["set", "claim_id", "exp_decision", "got_decision", "exp_payout", "got_payout", "both_ok"]]


Model: openai/gpt-6-luna | reasoning: high | 9 claims | 0 failed calls
Decision correct:        88.9%
Payout correct:          88.9%
Decision and payout ok:  77.8%
Total cost: $0.0050 | avg 11.1s per claim
             decision_ok  payout_ok  both_ok
set                                         
even_harder          0.5        1.0      0.5
harder               1.0        0.5      0.5
held_out             1.0        1.0      1.0
queue                1.0        1.0      1.0
security             1.0        1.0      1.0


,set,claim_id,exp_decision,got_decision,exp_payout,got_payout,both_ok
0,queue,CLM-1001,approve,approve,420.00,420.00,True
1,queue,CLM-1002,approve,approve,580.00,580.00,True
2,held_out,CLM-1101,approve,approve,580.00,580.00,True
3,held_out,CLM-1102,decline,decline,0.00,0.00,True
4,harder,CLM-1201,approve,approve,553.28,553.28,True
5,harder,CLM-1202,approve,approve,2048.44,2198.43,False
6,even_harder,CLM-1301,decline,refer,0.00,0.00,False
7,even_harder,CLM-1302,approve,approve,2473.00,2473.00,True
8,security,CLM-1901,decline,decline,0.00,0.00,True


In [86]:
results = evaluate_model("openai/gpt-6-luna", reasoning="low", limit=2)
results[["set", "claim_id", "exp_decision", "got_decision", "exp_payout", "got_payout", "both_ok"]]


Model: openai/gpt-6-luna | reasoning: low | 9 claims | 0 failed calls
Decision correct:        88.9%
Payout correct:          88.9%
Decision and payout ok:  77.8%
Total cost: $0.0026 | avg 5.8s per claim
             decision_ok  payout_ok  both_ok
set                                         
even_harder          0.5        1.0      0.5
harder               1.0        0.5      0.5
held_out             1.0        1.0      1.0
queue                1.0        1.0      1.0
security             1.0        1.0      1.0


,set,claim_id,exp_decision,got_decision,exp_payout,got_payout,both_ok
0,queue,CLM-1001,approve,approve,420.00,420.00,True
1,queue,CLM-1002,approve,approve,580.00,580.00,True
2,held_out,CLM-1101,approve,approve,580.00,580.00,True
3,held_out,CLM-1102,decline,decline,0.00,0.00,True
4,harder,CLM-1201,approve,approve,553.28,553.28,True
5,harder,CLM-1202,approve,approve,2048.44,2198.43,False
6,even_harder,CLM-1301,decline,refer,0.00,0.00,False
7,even_harder,CLM-1302,approve,approve,2473.00,2473.00,True
8,security,CLM-1901,decline,decline,0.00,0.00,True


In [88]:
from openrouter import OpenRouter  # pip install "openrouter>=1.2.1"

JEV = "typesafe/jev-1.13"
JEV_QUESTION = {
    "type": "choice",
    "instructions": "You are a claims handler at UnderCover Travel Insurance. Decide `claim` using `policy_wording` and the customer's `policy` record.",
    "criteria": {
        "approve": "The wording covers the claim and the required documents are attached.",
        "decline": "The wording does not cover the claim.",
        "refer": "A colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.",
    },
}

# Jev lives on OpenRouter's Decisions endpoint, which the OpenAI-style client can't reach
openrouter_client = OpenRouter(api_key=OPENROUTER_API_KEY, timeout_ms=60_000)


class JevDecision(BaseModel):
    """Jev only picks a decision: no rationale and no payout."""

    decision: Literal["approve", "decline", "refer"]
    confidence: float
    cost_usd: float
    seconds: float


def decide_with_jev(claim):
    """Ask Jev for the decision on one claim, with its confidence."""
    state = {
        "policy_wording": policy_wording,
        "policy": policies_by_id[claim["policy_id"]],
        "claim": claim,
    }
    start = time.perf_counter()
    response = openrouter_client.alpha.decisions.create(
        model=JEV, state=state, questions={"decision": JEV_QUESTION}
    )
    answer = response.answers["decision"]
    return JevDecision(
        decision=answer.choice,
        confidence=answer.confidence,
        cost_usd=response.usage.cost,
        seconds=round(time.perf_counter() - start, 2),
    )

In [89]:
jev_result = decide_with_jev(claims_by_id["CLM-1010"])
print(jev_result.model_dump_json(indent=2))

{
  "decision": "decline",
  "confidence": 0.56,
  "cost_usd": 0.000131586,
  "seconds": 0.75
}


In [ ]:
def evaluate_jev(sets=None, limit=None, workers=5):
    """Run Jev on every claim and score its decision against the handlers' decisions.

    Jev only picks a decision, so there is no payout to score.
    """

    def run(job):
        set_name, claim, exp = job
        row = {
            "set": set_name,
            "claim_id": claim["claim_id"],
            "exp_decision": exp["decision"],
            "got_decision": None,
            "confidence": float("nan"),
            "cost_usd": float("nan"),
            "seconds": float("nan"),
        }
        try:
            r = decide_with_jev(claim)
            row.update(got_decision=r.decision, confidence=r.confidence, cost_usd=r.cost_usd, seconds=r.seconds)
        except Exception as e:
            row.update(error=f"{type(e).__name__}: {e}")
        return row

    with ThreadPoolExecutor(workers) as pool:
        df = pd.DataFrame(list(pool.map(run, load_eval_jobs(sets, limit))))

    df["decision_ok"] = df["got_decision"] == df["exp_decision"]

    print(f"Model: {JEV} | {len(df)} claims | {int(df['got_decision'].isna().sum())} failed calls")
    print(f"Decision correct: {df['decision_ok'].mean():.1%}")
    print(f"Total cost: ${df['cost_usd'].sum():.4f} | avg {df['seconds'].mean():.2f}s per claim")
    confident = df["confidence"] >= 0.9
    for label, mask in (("Confidence >= 0.9", confident), ("Confidence <  0.9", ~confident)):
        share = f"{df.loc[mask, 'decision_ok'].mean():.1%}" if mask.any() else "n/a"
        print(f"{label}: {int(mask.sum())} claims, {share} correct")
    print(df.groupby("set")["decision_ok"].mean().round(2).to_string())
    return df

In [97]:
jev_results = evaluate_jev(limit=2)
jev_results[["set", "claim_id", "exp_decision", "got_decision", "confidence", "decision_ok"]]

Model: typesafe/jev-1.13 | 9 claims | 0 failed calls
Decision correct: 55.6%
Total cost: $0.0012 | avg 0.50s per claim
Confidence >= 0.9: 0 claims, n/a correct
Confidence <  0.9: 9 claims, 55.6% correct
set
even_harder    0.5
harder         0.5
held_out       0.5
queue          0.5
security       1.0


,set,claim_id,exp_decision,got_decision,confidence,decision_ok
0,queue,CLM-1001,approve,approve,0.85,True
1,queue,CLM-1002,approve,refer,0.40,False
2,held_out,CLM-1101,approve,approve,0.28,True
3,held_out,CLM-1102,decline,refer,0.11,False
4,harder,CLM-1201,approve,refer,0.22,False
5,harder,CLM-1202,approve,approve,0.37,True
6,even_harder,CLM-1301,decline,refer,0.09,False
7,even_harder,CLM-1302,approve,approve,0.46,True
8,security,CLM-1901,decline,decline,0.51,True


In [ ]:
from langchain_openai import ChatOpenAI


def make_llm(model="openrouter/free", reasoning=None, temperature=0):
    """Build a LangChain chat model that talks to OpenRouter."""
    return ChatOpenAI(
        model=model,
        base_url=OPENROUTER_BASE_URL,
        api_key=OPENROUTER_API_KEY,
        temperature=temperature,
        timeout=60,
        # OpenRouter-only options go in extra_body, e.g. the reasoning level
        extra_body={"reasoning": {"effort": reasoning}} if reasoning else None,
    )


# 1. Plain text


Hello!


In [100]:
llm = make_llm()
response = llm.invoke([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the meaning of life?."},
])
print(response.content)

That's one of humanity's oldest and biggest questions, and honestly, no one has a universally accepted answer!

The main perspectives people have landed on include:

- **Philosophers** like the existentialists argued that life has no built-in meaning, and we create our own through our choices and commitments.
- **Religious traditions** typically describe a purpose given by a higher power or a spiritual journey toward enlightenment.
- **Science** tends to frame meaning in terms of connection, love, and contribution to others.

Viktor Frankl, who survived the Holocaust, concluded that even in the worst suffering, meaning can be found through how we respond to our circumstances.

Many people find meaning in relationships, creativity, helping others, personal growth, or simply experiencing life deeply.

Perhaps the most honest answer is that meaning isn't something we discover like a buried treasure — it's something we build through what we love, value, and commit ourselves to.

What made 

In [101]:
from langchain_core.tools import tool


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record by policy ID (for example POL-2001): the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies_by_id.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"

In [102]:
print(get_policy.invoke({"policy_id": "POL-2001"}))
print(get_policy.invoke({"policy_id": "POL-0000"}))

{
  "policy_id": "POL-2001",
  "customer_id": "CUS-3001",
  "plan": "premium",
  "add_ons": [],
  "purchased_on": "2026-04-22",
  "sold_through": "Meridian Air checkout",
  "trip": {
    "destination": "Rome, Italy",
    "departs_on": "2026-06-10",
    "returns_on": "2026-06-18",
    "trip_cost": 3100.0
  },
  "travelers": [
    "Maya Patel"
  ],
  "status": "active"
}
There is no policy POL-0000


In [103]:
recorded = {}  # claim_id -> ClaimDecision
known_claim_ids = {
    c["claim_id"]
    for path in DATA_DIR.rglob("*claims*.json")
    for c in json.loads(path.read_text(encoding="utf-8"))
}


@tool
def record_decision(
    claim_id: str,
    decision: Literal["approve", "decline", "refer"],
    payout: float,
    rationale: str,
) -> str:
    """Record the final decision on a claim: its claim ID (for example CLM-1001), the decision (approve, decline or refer), the payout in US dollars (0 if declined or referred) and the rationale behind the decision."""
    if claim_id not in known_claim_ids:
        return f"There is no claim {claim_id}"
    recorded[claim_id] = ClaimDecision(rationale=rationale, decision=decision, payout=payout)
    return f"Recorded {decision} for {claim_id} with a payout of ${payout:,.2f}"

In [104]:
print(record_decision.invoke({
    "claim_id": "CLM-1001",
    "decision": "approve",
    "payout": 420.0,
    "rationale": "Covered under section 3. Eligible expenses are $245 + $95 + $80 = $420, below the $800 maximum.",
}))
print(record_decision.invoke({"claim_id": "CLM-0000", "decision": "decline", "payout": 0, "rationale": "n/a"}))
print(recorded["CLM-1001"].model_dump_json(indent=2))

Recorded approve for CLM-1001 with a payout of $420.00
There is no claim CLM-0000
{
  "rationale": "Covered under section 3. Eligible expenses are $245 + $95 + $80 = $420, below the $800 maximum.",
  "decision": "approve",
  "payout": 420.0
}


In [106]:
AGENT_INSTRUCTIONS = f"""You are a claims agent at UnderCover Travel Insurance, a travel insurance company.

You are given one claim (the customer's story, the items claimed and the supporting documents). Decide it using only the policy wording below.

Steps:
1. Call get_policy with the claim's policy ID to see the customer's plan, add-ons, trip dates and travelers.
2. Decide: approve (the claim is covered), decline (it is not covered) or refer (you cannot decide from the wording and the information you have, so a colleague must look at it).
3. Call record_decision exactly once with the claim ID, your decision, the payout and the rationale. The payout is 0 for a decline or a refer. Otherwise work it out with section 10 of the wording.

A valid claim is still an approve even when the payout is large. Do not guess: if a fact you need is missing or the wording does not settle the case, refer it. In the rationale, cite the section of the wording you relied on and show the calculation.

POLICY WORDING
{policy_wording}"""

In [107]:
def build_agent(llm, tools, system_prompt):
    """create_agent exists in LangChain 1.x; older versions have the same loop in LangGraph."""
    try:
        from langchain.agents import create_agent

        return create_agent(llm, tools, system_prompt=system_prompt)
    except ImportError:
        from langgraph.prebuilt import create_react_agent

        return create_react_agent(llm, tools, prompt=system_prompt)


def review_claim_with_agent(claim, model=DEFAULT_MODEL, reasoning=None):
    """Let an agent look up the policy, decide the claim and record the decision.

    Returns the recorded decision as a ClaimReview, with cost and time (None if the agent never
    recorded one), and the tools the agent called.
    """
    recorded.pop(claim["claim_id"], None)
    agent = build_agent(make_llm(model, reasoning), [get_policy, record_decision], AGENT_INSTRUCTIONS)
    start = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": json.dumps(claim, indent=2)}]})
    seconds = round(time.perf_counter() - start, 2)

    messages = result["messages"]
    tools_called = [m.name for m in messages if m.type == "tool"]
    ai_messages = [m for m in messages if m.type == "ai"]
    cost = sum((m.response_metadata.get("token_usage") or {}).get("cost") or 0 for m in ai_messages)
    decision = recorded.get(claim["claim_id"])
    if decision is None:
        return None, tools_called
    review = ClaimReview(
        **decision.model_dump(),
        cost_usd=cost,
        prompt_tokens=sum((m.usage_metadata or {}).get("input_tokens", 0) for m in ai_messages),
        completion_tokens=sum((m.usage_metadata or {}).get("output_tokens", 0) for m in ai_messages),
        seconds=seconds,
    )
    return review, tools_called

In [108]:
decision, tools_called = review_claim_with_agent(claims_by_id["CLM-1002"])
print("Tools called:", tools_called)
print(decision.model_dump_json(indent=2) if decision else "The agent did not record a decision")

Tools called: ['get_policy', 'record_decision']
{
  "rationale": "Covered under Section 5 (Baggage and Personal Effects). The loss occurred during the covered trip (June 20–29, 2026; incident June 21, 2026) and the carrier's property irregularity report (PRG-NW-48213) plus the settlement letter satisfy the Section 5 requirement to report the loss to the carrier and provide its report. Plan is Premium: Baggage and Personal Effects maximum $1,500, per-item limit $300, deductible $50.\n\nPer Section 10, eligible amount per item is the amount claimed or the per-item limit, whichever is lower. Per Section 5, a suit and a pair of shoes each count as one item:\n- Hard-shell suitcase $280 → $280\n- Wool suit $420 → capped at per-item limit $300\n- Dress shoes (pair) $160 → $160\n- Running shoes (pair) $110 → $110\n- Toiletry kit $60 → $60\n- Four shirts $120 → $120 (each shirt well under the limit)\nTotal eligible: $1,030.\n\nStep 2: subtract the $400 Northwind Airlines settlement for the same

In [ ]:
def evaluate_agent(model=DEFAULT_MODEL, reasoning=None, sets=None, limit=None, workers=5):
    """Run the agent on every claim, compare with the handlers' decisions and score it.

    Same arguments and scoring as evaluate_model. A claim where the agent never records a decision counts as wrong.
    """

    def run(job):
        set_name, claim, exp = job
        row = {
            "set": set_name,
            "claim_id": claim["claim_id"],
            "exp_decision": exp["decision"],
            "exp_payout": exp["payout"],
            # filled in on success; they stay empty if the call fails
            "got_decision": None,
            "got_payout": float("nan"),
            "cost_usd": float("nan"),
            "seconds": float("nan"),
        }
        try:
            r, tools_called = review_claim_with_agent(claim, model=model, reasoning=reasoning)
            row["tools_called"] = ", ".join(tools_called)
            if r is None:
                row["error"] = "the agent did not record a decision"
            else:
                row.update(got_decision=r.decision, got_payout=r.payout, cost_usd=r.cost_usd,
                           seconds=r.seconds, rationale=r.rationale)
        except Exception as e:
            row.update(error=f"{type(e).__name__}: {e}")
        return row

    with ThreadPoolExecutor(workers) as pool:
        df = pd.DataFrame(list(pool.map(run, load_eval_jobs(sets, limit))))

    df["decision_ok"] = df["got_decision"] == df["exp_decision"]
    df["payout_ok"] = (df["got_payout"] - df["exp_payout"]).abs() <= 0.01
    df["both_ok"] = df["decision_ok"] & df["payout_ok"]

    print(f"Agent | {model} | reasoning: {reasoning or 'default'} | {len(df)} claims | {int(df['got_decision'].isna().sum())} without a recorded decision")
    print(f"Decision correct:        {df['decision_ok'].mean():.1%}")
    print(f"Payout correct:          {df['payout_ok'].mean():.1%}")
    print(f"Decision and payout ok:  {df['both_ok'].mean():.1%}")
    print(f"Total cost: ${df['cost_usd'].sum():.4f} | avg {df['seconds'].mean():.1f}s per claim")
    print(df.groupby("set")[["decision_ok", "payout_ok", "both_ok"]].mean().round(2).to_string())
    return df

In [110]:
# Only the 25 claims in the queue
agent_results = evaluate_agent(sets=["queue"])
agent_results[["claim_id", "exp_decision", "got_decision", "exp_payout", "got_payout", "both_ok", "tools_called"]]

Agent | deepseek/deepseek-v4.1-flash-1017 | reasoning: default | 25 claims | 0 without a recorded decision
Decision correct:        76.0%
Payout correct:          68.0%
Decision and payout ok:  68.0%
Total cost: $0.0325 | avg 10.0s per claim
       decision_ok  payout_ok  both_ok
set                                   
queue         0.76       0.68     0.68


,claim_id,exp_decision,got_decision,exp_payout,got_payout,both_ok,tools_called
0,CLM-1001,approve,approve,420.0,420.0,True,"get_policy, record_decision"
1,CLM-1002,approve,approve,580.0,580.0,True,"get_policy, record_decision"
2,CLM-1003,approve,approve,3750.0,3750.0,True,"get_policy, record_decision"
3,CLM-1004,approve,approve,186.0,186.0,True,"get_policy, record_decision"
4,CLM-1005,approve,approve,243.0,243.0,True,"get_policy, record_decision"
5,CLM-1006,approve,approve,425.0,425.0,True,"get_policy, record_decision"
6,CLM-1007,approve,approve,50.0,50.0,True,"get_policy, record_decision"
7,CLM-1008,approve,approve,2240.0,2240.0,True,"get_policy, record_decision"
8,CLM-1009,approve,approve,930.0,930.0,True,"get_policy, record_decision"
9,CLM-1010,decline,decline,0.0,0.0,True,"get_policy, record_decision"


In [116]:
import contextlib
import io

TECHNIQUE_LABELS = {"llm": "LLM", "agent": "Agent", "jev": "Jev"}


def format_duration(seconds):
    minutes, secs = divmod(round(seconds), 60)
    return f"{minutes}m {secs:02d}s" if minutes else f"{secs}s"


def compare_experiments(experiments, sets=("queue",), limit=None, workers=5):
    """Run every experiment on the same claims and compare them.

    Each experiment is a dict: {"technique": "llm" | "agent" | "jev", "model": ..., "reasoning": ...}.
    Repeat an experiment to see how much its results vary. Jev takes no model or reasoning.
    sets / limit: which claims to run (default: the 25 in the queue).

    Returns (summary table, list of per-claim results). A claim is correct when the decision and the
    payout both match the handler's; Jev has no payout, so only its decision counts. Time is the wall-clock
    time of the whole experiment and cost is its total. The details of each claim are in the per-claim results.
    """
    rows, results = [], []
    for experiment in experiments:
        technique = experiment.get("technique", "llm")
        model = experiment.get("model", DEFAULT_MODEL)
        reasoning = experiment.get("reasoning")
        row = {
            "technique": TECHNIQUE_LABELS.get(technique, technique),
            "model": JEV if technique == "jev" else model,
            "reasoning": "n/a" if technique == "jev" else (reasoning or "default"),
        }
        start = time.perf_counter()
        try:
            with contextlib.redirect_stdout(io.StringIO()):  # hide each run's own printout
                if technique == "llm":
                    df = evaluate_model(model, reasoning, sets=sets, limit=limit, workers=workers)
                elif technique == "agent":
                    df = evaluate_agent(model, reasoning, sets=sets, limit=limit, workers=workers)
                elif technique == "jev":
                    df = evaluate_jev(sets=sets, limit=limit, workers=workers)
                else:
                    raise ValueError(f"Unknown technique {technique!r}: use llm, agent or jev")
            if df["got_decision"].isna().all():  # every call failed, so there is nothing to score
                raise RuntimeError(f"All calls failed. First error: {df['error'].iloc[0]}")
        except Exception as e:
            rows.append({**row, "error": f"{type(e).__name__}: {e}"})
            results.append(None)
            continue
        elapsed = time.perf_counter() - start

        correct = int(df["both_ok"].sum() if "both_ok" in df else df["decision_ok"].sum())
        failed = int(df["got_decision"].isna().sum())
        rows.append({
            **row,
            "accuracy": f"{correct / len(df):.0%}",
            "correct": f"{correct} of {len(df)}" + (f" ({failed} calls failed)" if failed else ""),
            "time": format_duration(elapsed),
            "cost": f"${df['cost_usd'].sum():.4f}",
        })
        results.append(df)
    return pd.DataFrame(rows), results

In [117]:
experiments = [
    {"technique": "jev"},
]

# limit=2 is a cheap trial; remove it to run all 25 claims of the queue for each experiment
summary, results = compare_experiments(experiments, limit=2)
summary

,technique,model,reasoning,accuracy,correct,time,cost
0,Jev,typesafe/jev-1.13,n/a,50%,1 of 2,1s,$0.0003


In [118]:
experiments = [

    {"technique": "llm", "model": "openai/gpt-6-luna", "reasoning": "low"},
        {"technique": "llm", "model": "openai/gpt-6-luna", "reasoning": "high"},
        {"technique": "llm", "model": "z-ai/glm-5.3-flash"},
        {"technique": "llm", "model": "anthropic/claude-haiku-5.5"},
        {"technique": "llm", "model": "moonshotai/kimi-k3"},
        {"technique": "llm", "model": "xiaomi/mimo-v2.6-flash"},
        {"technique": "llm", "agent": "google/gemini-2.5-flash-lite"},
        {"technique": "jev"},

]

# limit=2 is a cheap trial; remove it to run all 25 claims of the queue for each experiment
summary, results = compare_experiments(experiments, limit=25)
summary

,technique,model,reasoning,accuracy,correct,time,cost
0,LLM,openai/gpt-6-luna,low,60%,15 of 25,26s,$0.0063
1,LLM,openai/gpt-6-luna,high,76%,19 of 25,51s,$0.0111
2,LLM,z-ai/glm-5.3-flash,default,64%,16 of 25,1m 11s,$0.0113
3,LLM,anthropic/claude-haiku-5.5,default,68%,17 of 25,40s,$0.0251
4,LLM,moonshotai/kimi-k3,default,76%,19 of 25,2m 00s,$0.4280
5,LLM,xiaomi/mimo-v2.6-flash,default,56%,14 of 25,1m 23s,$0.0095
6,LLM,deepseek/deepseek-v4.1-flash-1017,default,72%,18 of 25,1m 18s,$0.0166
7,Jev,typesafe/jev-1.13,n/a,64%,16 of 25,3s,$0.0033
